In [1]:
import pandas as pd
import numpy as np

In [2]:
cancer_data=pd.read_csv(r"F:\Cancer AI Analyst\data\structured\processed\cancer_cleaned.csv")
cancer_data.head()

,Age,Gender,Country_Region,Year,Genetic_Risk,Air_Pollution,Alcohol_Use,Smoking,Obesity_Level,Cancer_Type,Cancer_Stage,Treatment_Cost_USD,Survival_Years,Target_Severity_Score
0,71,Male,UK,2021,6.4,2.8,9.5,0.9,8.7,Lung,Stage III,62913.44,5.9,4.92
1,34,Male,China,2021,1.3,4.5,3.7,3.9,6.3,Leukemia,Stage 0,12573.41,4.7,4.65
2,80,Male,Pakistan,2023,7.4,7.9,2.4,4.7,0.1,Breast,Stage II,6984.33,7.1,5.84
3,40,Male,UK,2015,1.7,2.9,4.8,3.5,2.7,Colon,Stage I,67446.25,1.6,3.12
4,43,Female,Brazil,2017,5.1,2.8,2.3,6.7,0.5,Skin,Stage III,77977.12,2.9,3.62


## Create Lifestyle Risk Score Where it is combining of smoking, Alcohol_Use, Obesity_Level, Air_Polution

In [5]:
cancer_data["Lifestyle_Risk_Score"]=(
    cancer_data["Smoking"]
    +
    cancer_data["Alcohol_Use"]
    +
    cancer_data["Obesity_Level"]
    +
    cancer_data["Air_Pollution"]
)
print(cancer_data["Lifestyle_Risk_Score"])

0        21.9
1        18.4
2        15.1
3        13.9
4        12.3
         ... 
49995    17.0
49996    25.2
49997    19.0
49998    30.8
49999    13.7
Name: Lifestyle_Risk_Score, Length: 50000, dtype: float64


## Create Overall Risk Score

In [6]:
cancer_data["Overall_Risk_Score"]=(
    cancer_data["Lifestyle_Risk_Score"]
    +
    cancer_data["Genetic_Risk"]
)
print(cancer_data["Overall_Risk_Score"])

0        28.3
1        19.7
2        22.5
3        15.6
4        17.4
         ... 
49995    19.3
49996    31.6
49997    25.2
49998    34.8
49999    18.8
Name: Overall_Risk_Score, Length: 50000, dtype: float64


## Create Age Category

In [8]:
cancer_data["Age_Group"]=pd.cut(
    cancer_data["Age"],
    bins=[0,40,60,80,100],
    labels=[
        "Young",
        "Middle_Aged",
        "Senior",
        "Elderly"
    ]
)

## Survival Category

In [9]:
cancer_data["Survival_Category"]=pd.cut(
    cancer_data["Survival_Years"],
    bins=[0,2,5,10,20],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very_High"
    ]
)

In [10]:
cancer_data["Cost_Category"]=pd.qcut(
    cancer_data["Treatment_Cost_USD"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very_High"
    ]
)

## Treatment Cost Category

In [11]:
cancer_data["Cost_Category"]=pd.qcut(
    cancer_data["Treatment_Cost_USD"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very_High"
    ]
)

## Stage Severity Mapping ---> Stage I = 1 ....

In [16]:
stage_map={
    "Stage I":1,
    "Stage II":2,
    "Stage III":3,
    "Stage IV":4
}
cancer_data["Stage_Severity"] = (
    cancer_data["Cancer_Stage"]
    .map(stage_map)
)

## Feature Importance Check

In [13]:
numerical_cols=cancer_data.select_dtypes(
    include=["int64","float64"]
).columns
corr=cancer_data[
    numerical_cols
].corr()
corr[
    "Target_Severity_Score"
].sort_values(
    ascending=False
)

Target_Severity_Score    1.000000
Overall_Risk_Score       0.866538
Lifestyle_Risk_Score     0.728396
Smoking                  0.484420
Genetic_Risk             0.478700
Air_Pollution            0.366963
Alcohol_Use              0.363250
Obesity_Level            0.251366
Survival_Years           0.004161
Year                     0.002109
Age                     -0.001481
Stage_Severity          -0.001893
Treatment_Cost_USD      -0.466058
Name: Target_Severity_Score, dtype: float64

In [17]:
cancer_data.head()

,Age,Gender,Country_Region,Year,Genetic_Risk,Air_Pollution,Alcohol_Use,Smoking,Obesity_Level,Cancer_Type,Cancer_Stage,Treatment_Cost_USD,Survival_Years,Target_Severity_Score,Lifestyle_Risk_Score,Overall_Risk_Score,Age_Group,Survival_Category,Cost_Category,Stage_Severity
0,71,Male,UK,2021,6.4,2.8,9.5,0.9,8.7,Lung,Stage III,62913.44,5.9,4.92,21.9,28.3,Senior,High,High,3.0
1,34,Male,China,2021,1.3,4.5,3.7,3.9,6.3,Leukemia,Stage 0,12573.41,4.7,4.65,18.4,19.7,Young,Medium,Low,NaN
2,80,Male,Pakistan,2023,7.4,7.9,2.4,4.7,0.1,Breast,Stage II,6984.33,7.1,5.84,15.1,22.5,Senior,High,Low,2.0
3,40,Male,UK,2015,1.7,2.9,4.8,3.5,2.7,Colon,Stage I,67446.25,1.6,3.12,13.9,15.6,Young,Low,High,1.0
4,43,Female,Brazil,2017,5.1,2.8,2.3,6.7,0.5,Skin,Stage III,77977.12,2.9,3.62,12.3,17.4,Middle_Aged,Medium,Very_High,3.0


## Save The Dataset

In [14]:
cancer_data.to_csv(
    "../data/structured/processed/cancer_feature_engineered.csv",
    index=False
)